# Meeting Intelligence Agent — Live on AWS Bedrock (Llama 3.3 70B)

**First real agent of the Technology Management Tower.** Runs against real meeting minutes, produces structured JSON, feeds every downstream agent.

> **Open this notebook directly in Colab:**
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snerantie/audit-slide-update/blob/real-agents/meeting-intelligence/tower/notebooks/01-meeting-intelligence.ipynb)

---

## Prerequisites (do these BEFORE running cells)

1. **AWS account** with Bedrock enabled — recommended region: `us-east-1`
2. **Bedrock model access** granted for **Meta Llama 3.3 70B Instruct**
   - AWS Console → **Bedrock** → **Model access** (left sidebar) → **Modify model access** → tick Meta Llama 3.3 70B Instruct → Save
   - Approval is instant for Meta models
3. **IAM permissions** on your user: `bedrock:InvokeModel`, `bedrock:Converse`, `bedrock:ListFoundationModels`
4. **AWS credentials ready to paste** — you'll need:
   - `AWS_ACCESS_KEY_ID`
   - `AWS_SECRET_ACCESS_KEY`
   - `AWS_SESSION_TOKEN` (only if using SSO/temporary credentials — otherwise leave blank)

## Cost expectation

Approximately **$0.005 per meeting** at Llama 3.3 70B pricing. Full run against 5 samples ≈ **$0.03 total**.

## 1 · One-cell setup

This cell:
- Detects Colab automatically
- Clones the repo (only on Colab; other environments skip this)
- Installs Python dependencies
- Sets up the Python import path

**Run it once.** Re-running is safe (idempotent).

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Detect Colab
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = 'real-agents/meeting-intelligence'
REPO_URL = 'https://github.com/snerantie/audit-slide-update.git'

if IN_COLAB:
    print('Google Colab detected')
    repo_dir = Path('/content/audit-slide-update')
    if not repo_dir.exists():
        print(f'Cloning branch: {BRANCH}')
        result = subprocess.run(
            ['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL],
            cwd='/content', check=True, capture_output=True, text=True,
        )
        print('Cloned.')
    else:
        print(f'Repo already at {repo_dir}')
    os.chdir(repo_dir / 'tower' / 'notebooks')
    print('Installing dependencies (30-60 seconds first time)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         'boto3>=1.34.0', 'botocore>=1.34.0', 'pydantic>=2.6.0', 'rich>=13.7.0'],
        check=True,
    )
    print('Dependencies ready.')
else:
    print('Not in Colab — assuming environment is set up manually.')
    print('If needed: `pip install -r ../requirements.txt` from the notebook dir.')

# Set Python path so we can `from src.xxx import ...`
TOWER_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd() / 'tower'
if str(TOWER_ROOT) not in sys.path:
    sys.path.insert(0, str(TOWER_ROOT))

sample_files = sorted((TOWER_ROOT / 'samples').glob('*.txt'))
print(f'\nWorking dir:  {os.getcwd()}')
print(f'Tower root:   {TOWER_ROOT}')
print(f'Samples ({len(sample_files)}):')
for f in sample_files:
    print(f'  {f.name}')

## 2 · Enter your AWS credentials

In Colab, credentials are entered via `getpass` and stored **only for this notebook session** — nothing is persisted or logged.

**If you already have valid credentials in the environment** (SageMaker Studio role, local `aws configure`, etc.) this cell will detect them and skip the prompt.

In [ ]:
from getpass import getpass

# ---- Config: edit if needed ----
AWS_REGION = 'eu-west-1'   # your region — 'eu-west-1' (Ireland) matches your setup

# Model ID — confirmed from Bedrock console (eu-west-1)
MODEL_ID   = 'zai.glm-4.7-flash'

# Alternatives if GLM fails (paste to try — verify spelling in your Bedrock console):
#   'eu.anthropic.claude-3-5-haiku-20241022-v1:0'     # Claude Haiku — reliable fallback
#   'eu.anthropic.claude-3-5-sonnet-20241022-v2:0'    # Claude Sonnet — best quality, pricier
#   'eu.meta.llama3-1-70b-instruct-v1:0'              # Llama 3.1 70B — if Meta available
#   'deepseek.v3-v1:0'                                # DeepSeek V3 — very cheap
#   'qwen.qwen3-32b-v1:0'                             # Qwen3 — cheap

# Only prompt for creds if not already in env (idempotent — safe to re-run)
if not os.environ.get('AWS_ACCESS_KEY_ID'):
    os.environ['AWS_ACCESS_KEY_ID'] = getpass('AWS_ACCESS_KEY_ID: ').strip()
if not os.environ.get('AWS_SECRET_ACCESS_KEY'):
    os.environ['AWS_SECRET_ACCESS_KEY'] = getpass('AWS_SECRET_ACCESS_KEY: ').strip()

# Session token — only needed for SSO/temporary credentials. Leave blank otherwise.
if 'AWS_SESSION_TOKEN' not in os.environ:
    token = getpass('AWS_SESSION_TOKEN (leave blank if not using SSO/temporary creds): ').strip()
    if token:
        os.environ['AWS_SESSION_TOKEN'] = token

print('\nCredentials set (values hidden).')
print(f'Region:   {AWS_REGION}')
print(f'Model ID: {MODEL_ID}')

## 3 · Connect to Bedrock + verify model access + smoke test

Three checks in one:
1. Boto3 clients created (STS identity confirmed)
2. Model access confirmed (Llama 3.3 70B is enabled)
3. Smoke test — one round-trip call that should return `OK`

If any of these fail, the error message will tell you exactly what to fix.

In [ ]:
from src.bedrock_client import get_bedrock_clients, verify_model_access, smoke_test

bedrock_runtime, bedrock = get_bedrock_clients(region=AWS_REGION)
print()
verify_model_access(bedrock, MODEL_ID)
print()
print('Running smoke test...')
result = smoke_test(bedrock_runtime, MODEL_ID)
print(f'Response: {result!r}')
assert 'OK' in result.upper(), 'Bedrock is not returning expected output'
print('\nBedrock is live. Ready to run the agent.')

## 4 · Load a sample meeting minute

Cyber MANCO from 8 July 2026 — the hero meeting from the demo. Real Northwind minute, ~2,000 words, 3 decisions, 2 risks, 9 actions.

In [ ]:
sample_path = TOWER_ROOT / 'samples' / '01-cyber-manco-2026-07-08.txt'
minute_text = sample_path.read_text()

print(f'Loaded: {sample_path.name}')
print(f'Size:   {len(minute_text):,} chars (~{len(minute_text)//4:,} tokens)')
print()
print('First 400 chars:')
print('-' * 60)
print(minute_text[:400])
print('...')

## 5 · Run the Meeting Intelligence Agent

Real Bedrock call. Real Llama 3.3 70B. Real structured extraction. Expect 3-8 seconds.

In [ ]:
from src.agents.meeting_intelligence import extract_meeting

record, meta = extract_meeting(
    minute_text=minute_text,
    bedrock_runtime=bedrock_runtime,
    model_id=MODEL_ID,
    verbose=False,
)

print(f'Extracted in {meta.latency_ms/1000:.1f}s')
print(f'  Tokens:  {meta.input_tokens:,} in / {meta.output_tokens:,} out')
print(f'  Cost:    ${meta.estimated_cost_usd:.5f}')
print(f'  Retries: {meta.retries}')
print()
print(f'  Forum:       {record.forum}')
print(f'  Date:        {record.date_iso}')
print(f'  Attendees:   {len(record.attendees)}')
print(f'  Decisions:   {len(record.decisions)}')
print(f'  Risks:       {len(record.risks)}')
print(f'  Actions:     {len(record.actions)}')
print(f'  Discussions: {len(record.discussions)}')

## 6 · Display the structured output

In [ ]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()
console.print(Panel(record.executive_summary, title='Executive Summary', border_style='cyan'))

In [ ]:
# Decisions
t = Table(title='Decisions', box=box.ROUNDED, show_lines=False)
t.add_column('ID', style='dim', width=6)
t.add_column('Decision', style='white')
t.add_column('Owner', style='green')
for d in record.decisions:
    t.add_row(d.id, d.text, d.owned_by or '—')
console.print(t)

In [ ]:
# Risks
t = Table(title='Risks', box=box.ROUNDED)
t.add_column('ID', style='dim', width=10)
t.add_column('Title', style='white')
t.add_column('Rating', style='red')
t.add_column('Change', style='yellow')
for r in record.risks:
    change = ''
    if r.is_uplifted:
        change = f'{r.from_rating or "?"} → {r.rating or "?"} (uplift)'
    elif r.is_new:
        change = 'NEW'
    t.add_row(r.id, r.title[:60], r.rating or '—', change)
console.print(t)

In [ ]:
# Actions
t = Table(title='Actions', box=box.ROUNDED, show_lines=True)
t.add_column('ID', style='dim', width=4)
t.add_column('Action', style='white', width=48)
t.add_column('Owner', style='green', width=18)
t.add_column('Due', style='yellow', width=14)
t.add_column('Pri', style='red', width=8)
for a in record.actions:
    t.add_row(
        a.id,
        a.text[:45] + ('...' if len(a.text) > 45 else ''),
        a.owner_candidate,
        a.due_label,
        a.priority.value,
    )
console.print(t)

## 7 · Save the structured output as JSON

In [ ]:
output_dir = TOWER_ROOT / 'output'
output_dir.mkdir(exist_ok=True)
out_path = output_dir / f'{sample_path.stem}.json'
out_path.write_text(record.model_dump_json(indent=2))
print(f'Written: {out_path}')
print(f'Size:    {out_path.stat().st_size:,} bytes')
print()
print('First 500 chars of the JSON:')
print('-' * 60)
print(out_path.read_text()[:500])

## 8 · Run against all 5 samples

Your evaluation baseline. Each run costs ~$0.005.

In [ ]:
from time import time

all_samples = sorted((TOWER_ROOT / 'samples').glob('*.txt'))
results = []

for path in all_samples:
    text = path.read_text()
    print(f'  {path.name}... ', end='', flush=True)
    started = time()
    try:
        rec, m = extract_meeting(text, bedrock_runtime, MODEL_ID)
        elapsed = time() - started
        print(f'{elapsed:.1f}s | {len(rec.actions)}A / {len(rec.risks)}R / {len(rec.decisions)}D | ${m.estimated_cost_usd:.4f}')
        (output_dir / f'{path.stem}.json').write_text(rec.model_dump_json(indent=2))
        results.append({'file': path.name, 'record': rec, 'meta': m, 'ok': True})
    except Exception as e:
        elapsed = time() - started
        print(f'FAILED after {elapsed:.1f}s: {type(e).__name__}: {e}')
        results.append({'file': path.name, 'error': str(e), 'ok': False})

print()
total_cost = sum(r['meta'].estimated_cost_usd for r in results if r['ok'])
total_latency = sum(r['meta'].latency_ms for r in results if r['ok'])
ok_count = sum(1 for r in results if r['ok'])
print(f'Summary:      {ok_count}/{len(results)} succeeded')
print(f'Total cost:   ${total_cost:.4f}')
print(f'Total time:   {total_latency/1000:.1f}s')

## 9 · Golden-set evaluation

The Cyber MANCO sample is hand-labelled — expected 3 decisions, 2 risks, 9 actions. Let's see how close the agent got.

In [ ]:
import json

golden_path = TOWER_ROOT / 'golden_set' / '01-cyber-manco-2026-07-08.expected.json'
golden = json.loads(golden_path.read_text())

cyber_result = next(r for r in results if r['file'].startswith('01-cyber') and r['ok'])
cyber = cyber_result['record']

checks = [
    ('forum name contains "Cyber"', 'Cyber' in cyber.forum),
    (f'date == {golden["date_iso"]}', cyber.date_iso == golden['date_iso']),
    (f'attendee count within 1 of {golden["attendee_count"]}', abs(len([a for a in cyber.attendees if a.attended]) - golden['attendee_count']) <= 1),
    (f'agenda item count within 1 of {golden["agenda_item_count"]}', abs(len(cyber.agenda) - golden['agenda_item_count']) <= 1),
    (f'decision count within 1 of {golden["decision_count"]}', abs(len(cyber.decisions) - golden['decision_count']) <= 1),
    (f'risk count within 1 of {golden["risk_count"]}', abs(len(cyber.risks) - golden['risk_count']) <= 1),
    (f'action count within 2 of {golden["action_count"]}', abs(len(cyber.actions) - golden['action_count']) <= 2),
    ('R-041 detected as uplifted', any('041' in r.id and r.is_uplifted for r in cyber.risks)),
]

for label, passed in checks:
    marker = 'PASS' if passed else 'FAIL'
    print(f'  [{marker}] {label}')

score = sum(1 for _, p in checks if p) / len(checks)
print(f'\nOverall: {score:.0%} ({sum(1 for _, p in checks if p)}/{len(checks)})')

## 10 · The hard case — a meeting with no explicit agenda

Real Teams meetings rarely have a clean agenda block in the transcript. The chair jumps in, topics emerge, actions get stated conversationally ("I'll have that with you by Friday"). This sample is that reality — a chaotic Payments review with **no explicit agenda**.

The agent should:
1. Infer topic segments (`agenda_inferred=true`)
2. Extract informal commitments as actions (e.g. "I'll come back to you by Friday")
3. Convert relative dates ("Friday 17 July", "end of this week") to ISO dates
4. Detect the new emerging risk (Data Platform resource contention)

In [ ]:
no_agenda_path = TOWER_ROOT / 'samples' / '06-payments-review-no-agenda.txt'
no_agenda_text = no_agenda_path.read_text()

print(f'Loaded: {no_agenda_path.name} ({len(no_agenda_text):,} chars)')
print()
chaotic_rec, chaotic_meta = extract_meeting(no_agenda_text, bedrock_runtime, MODEL_ID)

print(f'Extracted in {chaotic_meta.latency_ms/1000:.1f}s · ${chaotic_meta.estimated_cost_usd:.5f}')
print()
print(f'Forum:            {chaotic_rec.forum}')
print(f'Agenda inferred:  {chaotic_rec.agenda_inferred}  <- key: reconstructed from flow')
print(f'Agenda items ({len(chaotic_rec.agenda)}):')
for i, item in enumerate(chaotic_rec.agenda, 1):
    print(f'   {i}. {item}')
print()
print(f'Actions:  {len(chaotic_rec.actions)}')
print(f'Risks:    {len(chaotic_rec.risks)}')
print(f'Decisions: {len(chaotic_rec.decisions)}')

# Save output
(output_dir / f'{no_agenda_path.stem}.json').write_text(chaotic_rec.model_dump_json(indent=2))

In [ ]:
# Actions from the chaotic meeting — did the agent infer topics correctly?
t = Table(title='Actions from unstructured meeting', box=box.ROUNDED, show_lines=True)
t.add_column('ID', style='dim', width=4)
t.add_column('Action', style='white', width=42)
t.add_column('Owner', style='green', width=18)
t.add_column('Due (ISO)', style='cyan', width=12)
t.add_column('Topic', style='magenta', width=25)
for a in chaotic_rec.actions:
    t.add_row(
        a.id,
        a.text[:40] + ('...' if len(a.text) > 40 else ''),
        a.owner_candidate,
        a.due_date_iso or a.due_label[:12],
        (a.agenda_item or '—')[:23],
    )
console.print(t)

# Check the new risk was captured
print('\nNew risks captured:')
for r in chaotic_rec.risks:
    if r.is_new:
        print(f'   {r.id} · {r.title}')
        print(f'       driver: {r.driver}')

## What just happened?

You ran a real AWS Bedrock agent against 5 real (fictional-but-realistic) governance meeting minutes and extracted structured data that:

- Is validated against a Pydantic schema on the way out
- Carries a `source_quote` citation for every extracted item — end-to-end auditability
- Cost approximately **$0.03 total** for all 5 samples
- Runs in **~15-30 seconds** end to end

**This is Agent #1 of 9.** Its output feeds every downstream agent.

## What's next

### Agent #2 — Action Extraction Agent
Takes `MeetingRecord.actions[]` from above and enriches each with:
- Owner resolution against a directory (Entra ID / LDAP)
- Due date parsing to ISO format
- Duplicate detection against existing Jira tickets
- Confidence grading (A / B / C)

### Agent #3 — Jira Administration Agent
Takes enriched actions and creates real Jira tickets via the Atlassian REST API. Adds source citations, cross-links, HITL confirmation.

### Iteration ideas for Agent #1
1. Refine the system prompt in `src/agents/meeting_intelligence.py` — try adding few-shot examples
2. Test with a real (redacted) minute from your organisation — drop it in `samples/` and run
3. Add Claude 3.5 Sonnet as a verifier — reflect-and-verify pattern
4. Extend the Pydantic schema — add sentiment, agenda-item overrun, dependency edges